In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsgr18smartdata")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/stores.csv"

In [0]:
stores_schema = StructType(fields=[
    StructField("store_nbr", IntegerType(), False),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("type", StringType(), True),
    StructField("cluster", IntegerType(), True)
])

In [0]:
df_stores = spark.read \
    .schema(stores_schema) \
    .option("header", True) \
    .csv(ruta)

In [0]:
# Column order must match catalog_au.bronze.stores (insertInto is positional)
stores_final_df = df_stores.select(
    col("store_nbr"),
    col("city"),
    col("state"),
    col("type"),
    col("cluster")
)

In [0]:
stores_final_df.write \
    .mode("overwrite") \
    .insertInto(f"{catalogo}.{esquema}.stores")